In [1]:
from __future__ import annotations

import os
import random
import copy
import time
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score

# ============================================================
# 0. Configuration
# ============================================================
SEED = 258

DATA_ROOT = "/content/drive/MyDrive/Colab Notebooks/datasets/UCI-HAR"
OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/HAR/BG-HAR/UCI-HAR/ablation"

BATCH_SIZE = 128
EPOCHS = 80
LR = 2e-3
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 5.0

HIDDEN_DIM = 64
TOTAL_MAMBA_BLOCKS = 3
EARLY_EXIT_BLOCKS = 1
DROPOUT = 0.05
GATE_HIDDEN_DIM = 32

# default loss weights for Proposed
EARLY_LOSS_WEIGHT_DEFAULT = 0.30
GATE_LOSS_WEIGHT_DEFAULT = 0.35
COMPUTE_PENALTY_WEIGHT_DEFAULT = 0.01

BENEFIT_MARGIN = 0.07
MAX_FULL_ROUTE_RATIO = 0.45
VAL_RATIO = 0.20

NUM_WORKERS = 2
LATENCY_WARMUP_BATCHES = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ACTIVITY_NAMES = ["WALK", "UP", "DOWN", "SIT", "STAND", "LAY"]
SIGNAL_FILES = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]

# Ablation list
ABLATIONS = [
    {
        "name": "Proposed",
        "gate_mode": "full",
        "shuffle_pseudo_label": False,
        "early_loss_weight": EARLY_LOSS_WEIGHT_DEFAULT,
        "gate_loss_weight": GATE_LOSS_WEIGHT_DEFAULT,
        "compute_penalty_weight": COMPUTE_PENALTY_WEIGHT_DEFAULT,
    },
    # Full-route only / Early-route only are evaluated from the trained Proposed model.
    # They are added automatically to the final table.
    {
        "name": "w/o Compute penalty",
        "gate_mode": "full",
        "shuffle_pseudo_label": False,
        "early_loss_weight": EARLY_LOSS_WEIGHT_DEFAULT,
        "gate_loss_weight": GATE_LOSS_WEIGHT_DEFAULT,
        "compute_penalty_weight": 0.0,
    },
    {
        "name": "w/o Early loss",
        "gate_mode": "full",
        "shuffle_pseudo_label": False,
        "early_loss_weight": 0.0,
        "gate_loss_weight": GATE_LOSS_WEIGHT_DEFAULT,
        "compute_penalty_weight": COMPUTE_PENALTY_WEIGHT_DEFAULT,
    },
    {
        "name": "Confidence-only",
        "gate_mode": "confidence_only",
        "shuffle_pseudo_label": False,
        "early_loss_weight": EARLY_LOSS_WEIGHT_DEFAULT,
        "gate_loss_weight": GATE_LOSS_WEIGHT_DEFAULT,
        "compute_penalty_weight": COMPUTE_PENALTY_WEIGHT_DEFAULT,
    },
    {
        "name": "Representation-only",
        "gate_mode": "representation_only",
        "shuffle_pseudo_label": False,
        "early_loss_weight": EARLY_LOSS_WEIGHT_DEFAULT,
        "gate_loss_weight": GATE_LOSS_WEIGHT_DEFAULT,
        "compute_penalty_weight": COMPUTE_PENALTY_WEIGHT_DEFAULT,
    },
    {
        "name": "Shuffled pseudo-label",
        "gate_mode": "full",
        "shuffle_pseudo_label": True,
        "early_loss_weight": EARLY_LOSS_WEIGHT_DEFAULT,
        "gate_loss_weight": GATE_LOSS_WEIGHT_DEFAULT,
        "compute_penalty_weight": COMPUTE_PENALTY_WEIGHT_DEFAULT,
    },
]

# ============================================================
# 1. Reproducibility
# ============================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# ============================================================
# 2. Data loading
# ============================================================
def load_txt_matrix(path: str):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Required file not found:\n{path}")
    return pd.read_csv(path, sep=r"\s+", header=None).values.astype(np.float32)


def load_split(root: str, split: str):
    signal_dir = os.path.join(root, split, "Inertial Signals")
    signals = []
    for name in SIGNAL_FILES:
        file_path = os.path.join(signal_dir, f"{name}_{split}.txt")
        signals.append(load_txt_matrix(file_path))
    X = np.stack(signals, axis=1)  # (N, C, T)
    y = load_txt_matrix(os.path.join(root, split, f"y_{split}.txt")).astype(np.int64).reshape(-1) - 1
    subject = load_txt_matrix(os.path.join(root, split, f"subject_{split}.txt")).astype(np.int64).reshape(-1)
    return X, y, subject


def subject_disjoint_train_val_split(X, y, subject, val_ratio=0.20, seed=42):
    unique_subjects = np.array(sorted(np.unique(subject)))
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_subjects)
    n_val = max(1, int(round(len(unique_subjects) * val_ratio)))
    val_subjects = set(unique_subjects[:n_val].tolist())
    val_mask = np.array([s in val_subjects for s in subject])
    train_mask = ~val_mask
    return (
        X[train_mask], y[train_mask], subject[train_mask],
        X[val_mask], y[val_mask], subject[val_mask],
        sorted(list(val_subjects)),
    )


class HARWindowDataset(Dataset):
    def __init__(self, X, y, subject=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.subject = torch.zeros(len(y), dtype=torch.long) if subject is None else torch.tensor(subject, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.subject[idx]


def prepare_loaders():
    X_train_full, y_train_full, subject_train_full = load_split(DATA_ROOT, "train")
    X_test, y_test, subject_test = load_split(DATA_ROOT, "test")

    X_train, y_train, subject_train, X_val, y_val, subject_val, val_subjects = subject_disjoint_train_val_split(
        X_train_full, y_train_full, subject_train_full, val_ratio=VAL_RATIO, seed=SEED
    )

    train_mean = X_train.mean(axis=(0, 2), keepdims=True)
    train_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
    X_train = (X_train - train_mean) / train_std
    X_val = (X_val - train_mean) / train_std
    X_test = (X_test - train_mean) / train_std

    train_dataset = HARWindowDataset(X_train, y_train, subject_train)
    val_dataset = HARWindowDataset(X_val, y_val, subject_val)
    test_dataset = HARWindowDataset(X_test, y_test, subject_test)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    info = {
        "num_channels": X_train.shape[1],
        "seq_len": X_train.shape[2],
        "num_classes": len(ACTIVITY_NAMES),
        "val_subjects": val_subjects,
        "train_mean": train_mean,
        "train_std": train_std,
    }
    return train_loader, val_loader, test_loader, info

# ============================================================
# 3. Model
# ============================================================
class MiniMambaBlock(nn.Module):
    def __init__(self, hidden_dim: int, dropout: float = 0.2, conv_kernel: int = 5):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim)
        self.in_proj = nn.Linear(hidden_dim, hidden_dim * 2)
        self.depthwise_conv = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=conv_kernel,
                                        padding=conv_kernel // 2, groups=hidden_dim, bias=True)
        self.dt_proj = nn.Linear(hidden_dim, hidden_dim)
        self.b_proj = nn.Linear(hidden_dim, hidden_dim)
        self.c_proj = nn.Linear(hidden_dim, hidden_dim)
        self.A_log = nn.Parameter(torch.zeros(hidden_dim))
        self.D = nn.Parameter(torch.ones(hidden_dim))
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def selective_scan(self, u):
        B, T, H = u.shape
        dt = F.softplus(self.dt_proj(u)) + 1e-4
        b_t = torch.tanh(self.b_proj(u))
        c_t = torch.tanh(self.c_proj(u))
        A = torch.exp(self.A_log).view(1, H)
        state = torch.zeros(B, H, device=u.device, dtype=u.dtype)
        outputs = []
        for t in range(T):
            dt_cur = dt[:, t, :]
            u_cur = u[:, t, :]
            b_cur = b_t[:, t, :]
            c_cur = c_t[:, t, :]
            a_bar = torch.exp(-dt_cur * A)
            b_bar = (1.0 - a_bar) * b_cur
            state = a_bar * state + b_bar * u_cur
            y_cur = c_cur * state + self.D.view(1, H) * u_cur
            outputs.append(y_cur)
        return torch.stack(outputs, dim=1)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        u, z = self.in_proj(x).chunk(2, dim=-1)
        u = self.depthwise_conv(u.transpose(1, 2)).transpose(1, 2)
        u = F.silu(u)
        y = self.selective_scan(u)
        y = y * F.silu(z)
        y = self.out_proj(y)
        y = self.dropout(y)
        return residual + y


class BenefitGatedSharedMambaHAR(nn.Module):
    def __init__(self, in_channels: int, num_classes: int, hidden_dim: int = 64,
                 total_blocks: int = 3, early_exit_blocks: int = 1, dropout: float = 0.2,
                 gate_hidden_dim: int = 32, gate_mode: str = "full"):
        super().__init__()
        assert 1 <= early_exit_blocks < total_blocks
        assert gate_mode in ["full", "confidence_only", "representation_only"]

        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        self.total_blocks = total_blocks
        self.early_exit_blocks = early_exit_blocks
        self.gate_mode = gate_mode

        self.input_proj = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=1, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
        )
        self.blocks = nn.ModuleList([
            MiniMambaBlock(hidden_dim=hidden_dim, dropout=dropout, conv_kernel=5)
            for _ in range(total_blocks)
        ])
        self.readout_norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes))

        if gate_mode == "full":
            gate_input_dim = hidden_dim + num_classes + 6
        elif gate_mode == "confidence_only":
            gate_input_dim = 1
        else:  # representation_only
            gate_input_dim = hidden_dim

        self.benefit_gate = nn.Sequential(
            nn.LayerNorm(gate_input_dim),
            nn.Linear(gate_input_dim, gate_hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(gate_hidden_dim, 1),
        )

    def pool_feature(self, z):
        z = self.readout_norm(z)
        return z.mean(dim=1)

    def classify_from_feature(self, h):
        return self.classifier(h)

    def build_gate_input(self, x, h_early, early_logits):
        prob = F.softmax(early_logits, dim=1)
        top2 = torch.topk(prob, k=2, dim=1).values
        confidence = top2[:, 0:1]

        if self.gate_mode == "confidence_only":
            return confidence
        if self.gate_mode == "representation_only":
            return h_early

        margin = top2[:, 0:1] - top2[:, 1:2]
        entropy = -(prob * torch.log(prob + 1e-8)).sum(dim=1, keepdim=True)
        entropy = entropy / math.log(self.num_classes)
        temporal_energy = (x[:, :, 1:] - x[:, :, :-1]).pow(2).mean(dim=(1, 2)).unsqueeze(1)
        signal_energy = x.pow(2).mean(dim=(1, 2)).unsqueeze(1)
        abs_mean = x.abs().mean(dim=(1, 2)).unsqueeze(1)
        return torch.cat([h_early, early_logits, confidence, margin, entropy,
                          temporal_energy, signal_energy, abs_mean], dim=1)

    def forward_all(self, x):
        z = self.input_proj(x).transpose(1, 2)
        early_logits, h_early = None, None
        for i, block in enumerate(self.blocks):
            z = block(z)
            if i + 1 == self.early_exit_blocks:
                h_early = self.pool_feature(z)
                early_logits = self.classify_from_feature(h_early)
        gate_input = self.build_gate_input(x, h_early, early_logits)
        gate_logit = self.benefit_gate(gate_input).squeeze(1)
        gate_prob = torch.sigmoid(gate_logit)
        h_final = self.pool_feature(z)
        final_logits = self.classify_from_feature(h_final)
        return early_logits, final_logits, gate_logit, gate_prob

    @torch.no_grad()
    def forward_dynamic(self, x, benefit_tau: float):
        z = self.input_proj(x).transpose(1, 2)
        for i in range(self.early_exit_blocks):
            z = self.blocks[i](z)
        h_early = self.pool_feature(z)
        early_logits = self.classify_from_feature(h_early)
        gate_input = self.build_gate_input(x, h_early, early_logits)
        gate_logit = self.benefit_gate(gate_input).squeeze(1)
        gate_prob = torch.sigmoid(gate_logit)
        full_mask = gate_prob >= benefit_tau
        output_logits = early_logits.clone()
        route = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        if full_mask.any():
            z_full = z[full_mask]
            for i in range(self.early_exit_blocks, self.total_blocks):
                z_full = self.blocks[i](z_full)
            h_full = self.pool_feature(z_full)
            output_logits[full_mask] = self.classify_from_feature(h_full)
            route[full_mask] = 1
        return output_logits, route, gate_prob

    @torch.no_grad()
    def forward_early_only(self, x):
        z = self.input_proj(x).transpose(1, 2)
        for i in range(self.early_exit_blocks):
            z = self.blocks[i](z)
        h = self.pool_feature(z)
        return self.classify_from_feature(h)

    @torch.no_grad()
    def forward_full_only(self, x):
        z = self.input_proj(x).transpose(1, 2)
        for block in self.blocks:
            z = block(z)
        h = self.pool_feature(z)
        return self.classify_from_feature(h)

# ============================================================
# 4. FLOPs
# ============================================================
def flops_linear(in_dim, out_dim):
    return 2 * in_dim * out_dim


def flops_conv1d(in_ch, out_ch, kernel_size, out_len, groups=1):
    return 2 * out_len * out_ch * (in_ch // groups) * kernel_size


def estimate_minimamba_block_flops(T, H, conv_kernel=5):
    return (
        5 * T * H +
        T * flops_linear(H, 2 * H) +
        flops_conv1d(H, H, conv_kernel, T, groups=H) +
        T * flops_linear(H, H) * 3 +
        35 * T * H +
        T * flops_linear(H, H) +
        T * H
    )


def estimate_gate_flops(H, K, gate_hidden_dim, gate_mode):
    if gate_mode == "full":
        gate_input_dim = H + K + 6
        scalar_feature_ops = 40 * K + 20
    elif gate_mode == "confidence_only":
        gate_input_dim = 1
        scalar_feature_ops = 40 * K
    else:
        gate_input_dim = H
        scalar_feature_ops = 0
    return scalar_feature_ops + 5 * gate_input_dim + flops_linear(gate_input_dim, gate_hidden_dim) + gate_hidden_dim + flops_linear(gate_hidden_dim, 1) + 5


def estimate_route_flops(C, T, H, K, total_blocks, early_blocks, gate_hidden_dim, gate_mode, conv_kernel=5):
    input_proj = flops_conv1d(C, H, kernel_size=1, out_len=T, groups=1)
    block = estimate_minimamba_block_flops(T=T, H=H, conv_kernel=conv_kernel)
    head = 5 * T * H + T * H + flops_linear(H, K)
    gate = estimate_gate_flops(H, K, gate_hidden_dim, gate_mode)
    early_route = input_proj + early_blocks * block + head + gate
    full_route_dynamic = early_route + (total_blocks - early_blocks) * block + head
    full_model_once = input_proj + total_blocks * block + head
    return {
        "early_route": early_route,
        "full_route_dynamic": full_route_dynamic,
        "full_model_once": full_model_once,
        "one_mamba_block": block,
        "benefit_gate": gate,
    }


def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

# ============================================================
# 5. Train / eval utils
# ============================================================
criterion_cls = nn.CrossEntropyLoss()
criterion_gate = nn.BCEWithLogitsLoss()


def make_benefit_target(early_logits, final_logits, y, benefit_margin=0.05, shuffle=False):
    with torch.no_grad():
        early_ce = F.cross_entropy(early_logits, y, reduction="none")
        final_ce = F.cross_entropy(final_logits, y, reduction="none")
        early_pred = early_logits.argmax(dim=1)
        final_pred = final_logits.argmax(dim=1)
        corrected_by_full = (early_pred != y) & (final_pred == y)
        meaningful_loss_gain = (early_ce - final_ce) > benefit_margin
        full_harms_early = (early_pred == y) & (final_pred != y)
        target = (corrected_by_full | (meaningful_loss_gain & (~full_harms_early))).float()
        if shuffle:
            target = target[torch.randperm(target.size(0), device=target.device)]
    return target


def train_one_epoch(model, loader, optimizer, loss_cfg):
    model.train()
    all_y, all_early, all_final = [], [], []
    total_loss = 0.0
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        early_logits, final_logits, gate_logit, gate_prob = model.forward_all(x)
        early_loss = criterion_cls(early_logits, y)
        final_loss = criterion_cls(final_logits, y)
        benefit_target = make_benefit_target(
            early_logits, final_logits, y,
            benefit_margin=BENEFIT_MARGIN,
            shuffle=loss_cfg["shuffle_pseudo_label"],
        )
        gate_loss = criterion_gate(gate_logit, benefit_target)
        compute_penalty = gate_prob.mean()
        loss = (
            final_loss +
            loss_cfg["early_loss_weight"] * early_loss +
            loss_cfg["gate_loss_weight"] * gate_loss +
            loss_cfg["compute_penalty_weight"] * compute_penalty
        )
        loss.backward()
        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        all_y.append(y.detach().cpu().numpy())
        all_early.append(early_logits.argmax(dim=1).detach().cpu().numpy())
        all_final.append(final_logits.argmax(dim=1).detach().cpu().numpy())
    all_y = np.concatenate(all_y)
    all_early = np.concatenate(all_early)
    all_final = np.concatenate(all_final)
    return {
        "loss": total_loss / len(loader.dataset),
        "early_macro_f1": f1_score(all_y, all_early, average="macro"),
        "final_macro_f1": f1_score(all_y, all_final, average="macro"),
    }


@torch.no_grad()
def evaluate_all(model, loader):
    model.eval()
    all_y, all_early, all_final, all_gate_prob = [], [], [], []
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        early_logits, final_logits, _, gate_prob = model.forward_all(x)
        all_y.append(y.numpy())
        all_early.append(early_logits.argmax(dim=1).detach().cpu().numpy())
        all_final.append(final_logits.argmax(dim=1).detach().cpu().numpy())
        all_gate_prob.append(gate_prob.detach().cpu().numpy())
    y_true = np.concatenate(all_y)
    early_pred = np.concatenate(all_early)
    final_pred = np.concatenate(all_final)
    gate_prob = np.concatenate(all_gate_prob)
    return {
        "y_true": y_true,
        "early_pred": early_pred,
        "final_pred": final_pred,
        "gate_prob": gate_prob,
        "early_acc": accuracy_score(y_true, early_pred),
        "early_macro_f1": f1_score(y_true, early_pred, average="macro"),
        "final_acc": accuracy_score(y_true, final_pred),
        "final_macro_f1": f1_score(y_true, final_pred, average="macro"),
    }


def dynamic_metrics_from_outputs(y_true, early_pred, final_pred, gate_prob, tau, flops_info):
    full_mask = gate_prob >= tau
    dynamic_pred = early_pred.copy()
    dynamic_pred[full_mask] = final_pred[full_mask]
    full_ratio = float(full_mask.mean())
    avg_flops = (1.0 - full_ratio) * flops_info["early_route"] + full_ratio * flops_info["full_route_dynamic"]
    return {
        "tau": float(tau),
        "macro_f1": f1_score(y_true, dynamic_pred, average="macro"),
        "acc": accuracy_score(y_true, dynamic_pred),
        "full_ratio": full_ratio,
        "avg_flops": avg_flops,
    }


def calibrate_tau(y_true, early_pred, final_pred, gate_prob, flops_info, max_full_route_ratio=0.45):
    candidate_taus = np.unique(np.concatenate([
        np.quantile(gate_prob, np.linspace(0.01, 0.99, 199)),
        np.array([gate_prob.min() - 1e-6, gate_prob.max() + 1e-6]),
    ]))
    records = [dynamic_metrics_from_outputs(y_true, early_pred, final_pred, gate_prob, tau, flops_info)
               for tau in candidate_taus]
    feasible = [r for r in records if r["full_ratio"] <= max_full_route_ratio]
    pool = feasible if len(feasible) > 0 else records
    best = sorted(pool, key=lambda r: (r["macro_f1"], -r["avg_flops"]), reverse=True)[0]
    return best, records


@torch.no_grad()
def evaluate_dynamic(model, loader, tau, flops_info):
    model.eval()
    all_y, all_pred, all_route = [], [], []
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        logits, route, _ = model.forward_dynamic(x, benefit_tau=tau)
        all_y.append(y.numpy())
        all_pred.append(logits.argmax(dim=1).detach().cpu().numpy())
        all_route.append(route.detach().cpu().numpy())
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    route = np.concatenate(all_route)
    full_ratio = float(route.mean())
    avg_flops = (1.0 - full_ratio) * flops_info["early_route"] + full_ratio * flops_info["full_route_dynamic"]
    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "acc": accuracy_score(y_true, y_pred),
        "full_ratio": full_ratio,
        "avg_flops": avg_flops,
    }


@torch.no_grad()
def evaluate_fixed_route(model, loader, route_name, flops_info):
    model.eval()
    all_y, all_pred = [], []
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        if route_name == "early":
            logits = model.forward_early_only(x)
        elif route_name == "full":
            logits = model.forward_full_only(x)
        else:
            raise ValueError(route_name)
        all_y.append(y.numpy())
        all_pred.append(logits.argmax(dim=1).detach().cpu().numpy())
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    if route_name == "early":
        full_ratio = 0.0
        avg_flops = flops_info["early_route"]
    else:
        full_ratio = 1.0
        avg_flops = flops_info["full_model_once"]
    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "acc": accuracy_score(y_true, y_pred),
        "full_ratio": full_ratio,
        "avg_flops": avg_flops,
    }

# ============================================================
# 6. Latency
# ============================================================
@torch.no_grad()
def measure_cpu_latency(model, loader, mode, tau=None, warmup=3):
    m = copy.deepcopy(model).cpu().eval()
    for i, (x, _, _) in enumerate(loader):
        x = x.cpu()
        if mode == "dynamic":
            _ = m.forward_dynamic(x, benefit_tau=tau)
        elif mode == "early":
            _ = m.forward_early_only(x)
        elif mode == "full":
            _ = m.forward_full_only(x)
        else:
            raise ValueError(mode)
        if i + 1 >= warmup:
            break
    total_time = 0.0
    total_samples = 0
    for x, _, _ in loader:
        x = x.cpu()
        t0 = time.perf_counter()
        if mode == "dynamic":
            _ = m.forward_dynamic(x, benefit_tau=tau)
        elif mode == "early":
            _ = m.forward_early_only(x)
        else:
            _ = m.forward_full_only(x)
        t1 = time.perf_counter()
        total_time += t1 - t0
        total_samples += x.size(0)
    return total_time / total_samples * 1000.0

# ============================================================
# 7. Main ablation runner
# ============================================================
def train_variant(cfg, loaders, data_info):
    train_loader, val_loader, test_loader = loaders
    set_seed(SEED)

    model = BenefitGatedSharedMambaHAR(
        in_channels=data_info["num_channels"],
        num_classes=data_info["num_classes"],
        hidden_dim=HIDDEN_DIM,
        total_blocks=TOTAL_MAMBA_BLOCKS,
        early_exit_blocks=EARLY_EXIT_BLOCKS,
        dropout=DROPOUT,
        gate_hidden_dim=GATE_HIDDEN_DIM,
        gate_mode=cfg["gate_mode"],
    ).to(DEVICE)

    flops_info = estimate_route_flops(
        C=data_info["num_channels"], T=data_info["seq_len"], H=HIDDEN_DIM, K=data_info["num_classes"],
        total_blocks=TOTAL_MAMBA_BLOCKS, early_blocks=EARLY_EXIT_BLOCKS,
        gate_hidden_dim=GATE_HIDDEN_DIM, gate_mode=cfg["gate_mode"], conv_kernel=5,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_state = None
    best_val_final_f1 = -1.0
    history = []

    for epoch in range(1, EPOCHS + 1):
        train_log = train_one_epoch(model, train_loader, optimizer, cfg)
        val_log = evaluate_all(model, val_loader)
        scheduler.step()

        history.append({
            "epoch": epoch,
            "train_loss": train_log["loss"],
            "train_early_macro_f1": train_log["early_macro_f1"],
            "train_final_macro_f1": train_log["final_macro_f1"],
            "val_early_macro_f1": val_log["early_macro_f1"],
            "val_final_macro_f1": val_log["final_macro_f1"],
        })

        if val_log["final_macro_f1"] > best_val_final_f1:
            best_val_final_f1 = val_log["final_macro_f1"]
            best_state = copy.deepcopy(model.state_dict())

        if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
            print(f"[{cfg['name']}] Epoch {epoch:03d} | Val early F1={val_log['early_macro_f1']:.4f} | Val full F1={val_log['final_macro_f1']:.4f}")

    model.load_state_dict(best_state)

    val_outputs = evaluate_all(model, val_loader)
    best_tau, tau_records = calibrate_tau(
        y_true=val_outputs["y_true"],
        early_pred=val_outputs["early_pred"],
        final_pred=val_outputs["final_pred"],
        gate_prob=val_outputs["gate_prob"],
        flops_info=flops_info,
        max_full_route_ratio=MAX_FULL_ROUTE_RATIO,
    )
    tau = best_tau["tau"]

    test_dynamic = evaluate_dynamic(model, test_loader, tau=tau, flops_info=flops_info)
    latency = measure_cpu_latency(model, test_loader, mode="dynamic", tau=tau, warmup=LATENCY_WARMUP_BATCHES)

    row = {
        "Variant": cfg["name"],
        "Macro-F1": test_dynamic["macro_f1"],
        "Full Ratio": test_dynamic["full_ratio"],
        "Avg.FLOPs (M)": test_dynamic["avg_flops"] / 1e6,
        "Latency": latency,
        "Params": count_params(model) / 1e6,
        "Tau": tau,
        "Val Macro-F1": best_tau["macro_f1"],
        "Val Full Ratio": best_tau["full_ratio"],
    }

    save_obj = {
        "model_state_dict": model.state_dict(),
        "config": cfg,
        "history": pd.DataFrame(history),
        "tau_records": pd.DataFrame(tau_records),
        "flops_info": flops_info,
        "result": row,
    }
    return model, flops_info, row, save_obj


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    print("Device:", DEVICE)

    train_loader, val_loader, test_loader, data_info = prepare_loaders()
    loaders = (train_loader, val_loader, test_loader)
    print("Train/Val/Test:", len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))
    print("Val subjects:", data_info["val_subjects"])

    final_rows = []
    all_save_objs = {}
    proposed_model = None
    proposed_flops = None

    for cfg in ABLATIONS:
        print("\n" + "=" * 80)
        print("Running:", cfg["name"])
        print("=" * 80)
        model, flops_info, row, save_obj = train_variant(cfg, loaders, data_info)
        final_rows.append(row)
        all_save_objs[cfg["name"]] = save_obj
        torch.save(save_obj, os.path.join(OUT_DIR, f"{cfg['name'].replace('/', 'wo').replace(' ', '_')}.pt"))

        if cfg["name"] == "Proposed":
            proposed_model = copy.deepcopy(model).to(DEVICE)
            proposed_flops = flops_info

    # Fixed-route baselines from Proposed model
    print("\n" + "=" * 80)
    print("Evaluating fixed-route baselines from Proposed model")
    print("=" * 80)

    full_res = evaluate_fixed_route(proposed_model, test_loader, route_name="full", flops_info=proposed_flops)
    full_latency = measure_cpu_latency(proposed_model, test_loader, mode="full", warmup=LATENCY_WARMUP_BATCHES)
    full_row = {
        "Variant": "Full-route only",
        "Macro-F1": full_res["macro_f1"],
        "Full Ratio": 1.0,
        "Avg.FLOPs (M)": full_res["avg_flops"] / 1e6,
        "Latency": full_latency,
        "Params": count_params(proposed_model) / 1e6,
        "Tau": np.nan,
        "Val Macro-F1": np.nan,
        "Val Full Ratio": np.nan,
    }

    early_res = evaluate_fixed_route(proposed_model, test_loader, route_name="early", flops_info=proposed_flops)
    early_latency = measure_cpu_latency(proposed_model, test_loader, mode="early", warmup=LATENCY_WARMUP_BATCHES)
    early_row = {
        "Variant": "Early-route only",
        "Macro-F1": early_res["macro_f1"],
        "Full Ratio": 0.0,
        "Avg.FLOPs (M)": early_res["avg_flops"] / 1e6,
        "Latency": early_latency,
        "Params": count_params(proposed_model) / 1e6,
        "Tau": np.nan,
        "Val Macro-F1": np.nan,
        "Val Full Ratio": np.nan,
    }

    # Desired table order
    table_order = [
        "Proposed", "Full-route only", "Early-route only",
        "w/o Compute penalty", "w/o Early loss",
        "Confidence-only", "Representation-only", "Shuffled pseudo-label",
    ]
    final_rows.extend([full_row, early_row])
    result_df = pd.DataFrame(final_rows)
    result_df["_order"] = result_df["Variant"].apply(lambda x: table_order.index(x))
    result_df = result_df.sort_values("_order").drop(columns=["_order"])

    # Rounded display table
    display_df = result_df.copy()
    for col in ["Macro-F1", "Full Ratio", "Avg.FLOPs (M)", "Latency", "Params", "Tau", "Val Macro-F1", "Val Full Ratio"]:
        display_df[col] = display_df[col].astype(float).round(4)

    csv_path = os.path.join(OUT_DIR, "ablation_results.csv")
    xlsx_path = os.path.join(OUT_DIR, "ablation_results.xlsx")
    result_df.to_csv(csv_path, index=False)
    display_df.to_excel(xlsx_path, index=False)

    torch.save({
        "all_results": result_df,
        "display_results": display_df,
        "all_save_objs": all_save_objs,
    }, os.path.join(OUT_DIR, "all_ablation_outputs.pt"))

    print("\n================ Ablation Study Results ================")
    print(display_df[["Variant", "Macro-F1", "Full Ratio", "Avg.FLOPs (M)", "Latency", "Params"]].to_string(index=False))
    print("\nSaved:")
    print(csv_path)
    print(xlsx_path)
    print(os.path.join(OUT_DIR, "all_ablation_outputs.pt"))


if __name__ == "__main__":
    main()

Device: cuda
Train/Val/Test: 5913 1439 2947
Val subjects: [1, 15, 23, 26]

Running: Proposed
[Proposed] Epoch 001 | Val early F1=0.8880 | Val full F1=0.9677
[Proposed] Epoch 010 | Val early F1=0.9318 | Val full F1=0.9450
[Proposed] Epoch 020 | Val early F1=0.9727 | Val full F1=0.9744
[Proposed] Epoch 030 | Val early F1=0.9577 | Val full F1=0.9830
[Proposed] Epoch 040 | Val early F1=0.9470 | Val full F1=0.9810
[Proposed] Epoch 050 | Val early F1=0.9733 | Val full F1=0.9728
[Proposed] Epoch 060 | Val early F1=0.9695 | Val full F1=0.9893
[Proposed] Epoch 070 | Val early F1=0.9752 | Val full F1=0.9887
[Proposed] Epoch 080 | Val early F1=0.9745 | Val full F1=0.9887

Running: w/o Compute penalty
[w/o Compute penalty] Epoch 001 | Val early F1=0.8888 | Val full F1=0.9677
[w/o Compute penalty] Epoch 010 | Val early F1=0.9328 | Val full F1=0.9490
[w/o Compute penalty] Epoch 020 | Val early F1=0.9377 | Val full F1=0.9472
[w/o Compute penalty] Epoch 030 | Val early F1=0.9719 | Val full F1=0.9766
[